In [9]:
import json

from pathlib import Path

import numpy as np
import pandas as pd

project_root = Path.cwd()

if project_root.name == 'notebooks':
    project_root = project_root.parent

structure_path = project_root / 'artifacts' / 'evaluation' / 'rgrag_structure_results.jsonl'
edu_path = project_root / 'artifacts' / 'evaluation' / 'rgrag_edu_results.jsonl'

In [10]:
def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as file:
        return [
            json.loads(line)
            for line in file
            if line.strip()
        ]

structure_df = pd.json_normalize(load_jsonl(structure_path))
edu_df = pd.json_normalize(load_jsonl(edu_path))

print('RGRAG-Structure queries:', len(structure_df))
print('RGRAG-EDU queries:', len(edu_df))

RGRAG-Structure queries: 2255
RGRAG-EDU queries: 2255


In [11]:
structure_ids = set(structure_df['dataset_index'])
edu_ids = set(edu_df['dataset_index'])

print('Same query set:', structure_ids == edu_ids)
print('Structure only:', len(structure_ids - edu_ids))
print('EDU only:', len(edu_ids - structure_ids))

Same query set: True
Structure only: 0
EDU only: 0


In [12]:
comparison = structure_df[
    [
        'dataset_index',
        'rgrag_candidate_recall',
        'rgrag_candidate_complete',
    ]
].merge(
    edu_df[
        [
            'dataset_index',
            'candidate_recall',
            'candidate_complete',
        ]
    ],
    on='dataset_index',
)

print(
    'Same Candidate Recall:',
    np.allclose(
        comparison['rgrag_candidate_recall'],
        comparison['candidate_recall'],
    ),
)

print(
    'Same Candidate Complete:',
    (
        comparison['rgrag_candidate_complete']
        == comparison['candidate_complete']
    ).all(),
)

Same Candidate Recall: True
Same Candidate Complete: True


In [13]:
metrics = [
    (
        'Candidate Recall',
        'graphrag_candidate_recall',
        'rgrag_candidate_recall',
        'candidate_recall',
    ),
    (
        'Candidate Complete Recall',
        'graphrag_candidate_complete',
        'rgrag_candidate_complete',
        'candidate_complete',
    ),
    (
        'Hits@4',
        'graphrag_metrics.hits_at_4',
        'rgrag_metrics.hits_at_4',
        'metrics.hits_at_4',
    ),
    (
        'Hits@10',
        'graphrag_metrics.hits_at_10',
        'rgrag_metrics.hits_at_10',
        'metrics.hits_at_10',
    ),
    (
        'MAP@10',
        'graphrag_metrics.map_at_10',
        'rgrag_metrics.map_at_10',
        'metrics.map_at_10',
    ),
    (
        'MRR@10',
        'graphrag_metrics.mrr_at_10',
        'rgrag_metrics.mrr_at_10',
        'metrics.mrr_at_10',
    ),
    (
        'Complete@4',
        'graphrag_metrics.complete_at_4',
        'rgrag_metrics.complete_at_4',
        'metrics.complete_at_4',
    ),
    (
        'Complete@10',
        'graphrag_metrics.complete_at_10',
        'rgrag_metrics.complete_at_10',
        'metrics.complete_at_10',
    ),
]

for name, graphrag_col, structure_col, edu_col in metrics:
    graphrag = structure_df[graphrag_col].mean()
    structure = structure_df[structure_col].mean()
    edu = edu_df[edu_col].mean()

    print()
    print(name)
    print('GraphRAG:       ', round(graphrag, 4))
    print('RGRAG-Structure:', round(structure, 4))
    print('RGRAG-EDU:      ', round(edu, 4))
    print('Structure gain: ', f'{structure - graphrag:+.4f}')
    print('EDU gain vs GR: ', f'{edu - graphrag:+.4f}')
    print('EDU vs Structure:', f'{edu - structure:+.4f}')


Candidate Recall
GraphRAG:        0.88
RGRAG-Structure: 0.917
RGRAG-EDU:       0.917
Structure gain:  +0.0370
EDU gain vs GR:  +0.0370
EDU vs Structure: +0.0000

Candidate Complete Recall
GraphRAG:        0.7854
RGRAG-Structure: 0.8412
RGRAG-EDU:       0.8412
Structure gain:  +0.0559
EDU gain vs GR:  +0.0559
EDU vs Structure: +0.0000

Hits@4
GraphRAG:        0.7659
RGRAG-Structure: 0.7787
RGRAG-EDU:       0.7322
Structure gain:  +0.0129
EDU gain vs GR:  -0.0337
EDU vs Structure: -0.0466

Hits@10
GraphRAG:        0.9024
RGRAG-Structure: 0.9184
RGRAG-EDU:       0.9086
Structure gain:  +0.0160
EDU gain vs GR:  +0.0062
EDU vs Structure: -0.0098

MAP@10
GraphRAG:        0.3116
RGRAG-Structure: 0.3169
RGRAG-EDU:       0.3053
Structure gain:  +0.0053
EDU gain vs GR:  -0.0063
EDU vs Structure: -0.0116

MRR@10
GraphRAG:        0.6045
RGRAG-Structure: 0.6135
RGRAG-EDU:       0.5769
Structure gain:  +0.0090
EDU gain vs GR:  -0.0276
EDU vs Structure: -0.0366

Complete@4
GraphRAG:        0.1392
RG

In [15]:
ranking_df = structure_df[
    [
        'dataset_index',
        'rgrag_metrics.map_at_10',
        'rgrag_metrics.mrr_at_10',
        'rgrag_metrics.complete_at_10',
    ]
].merge(
    edu_df[
        [
            'dataset_index',
            'metrics.map_at_10',
            'metrics.mrr_at_10',
            'metrics.complete_at_10',
        ]
    ],
    on='dataset_index',
)

ranking_df['map_gain'] = (
    ranking_df['metrics.map_at_10']
    - ranking_df['rgrag_metrics.map_at_10']
)

ranking_df['mrr_gain'] = (
    ranking_df['metrics.mrr_at_10']
    - ranking_df['rgrag_metrics.mrr_at_10']
)

In [16]:
print('MAP: RGRAG-EDU vs RGRAG-Structure')
print('Improved:', int((ranking_df['map_gain'] > 0).sum()), f'({(ranking_df["map_gain"] > 0).mean():.2%})')
print('Unchanged:', int((ranking_df['map_gain'] == 0).sum()), f'({(ranking_df["map_gain"] == 0).mean():.2%})')
print('Worsened:', int((ranking_df['map_gain'] < 0).sum()), f'({(ranking_df["map_gain"] < 0).mean():.2%})')

print()

print('MRR: RGRAG-EDU vs RGRAG-Structure')
print('Improved:', int((ranking_df['mrr_gain'] > 0).sum()), f'({(ranking_df["mrr_gain"] > 0).mean():.2%})')
print('Unchanged:', int((ranking_df['mrr_gain'] == 0).sum()), f'({(ranking_df["mrr_gain"] == 0).mean():.2%})')
print('Worsened:', int((ranking_df['mrr_gain'] < 0).sum()), f'({(ranking_df["mrr_gain"] < 0).mean():.2%})')

MAP: RGRAG-EDU vs RGRAG-Structure
Improved: 825 (36.59%)
Unchanged: 368 (16.32%)
Worsened: 1062 (47.10%)

MRR: RGRAG-EDU vs RGRAG-Structure
Improved: 527 (23.37%)
Unchanged: 951 (42.17%)
Worsened: 777 (34.46%)


In [17]:
import json

from collections import Counter

rst_edges_path = project_root / 'artifacts' / 'rgrag' / 'rst_relationship_edges.jsonl'

relations = Counter()
nuclearities = Counter()

with open(rst_edges_path, 'r', encoding='utf-8') as file:
    for line in file:
        line = line.strip()

        if not line:
            continue

        edge = json.loads(line)

        relations[edge['relation']] += 1
        nuclearities[edge['nuclearity']] += 1

print('RST RELATIONS')
print()

for relation, count in relations.most_common():
    print(f'{relation}: {count}')

print()
print('NUCLEARITY')
print()

for nuclearity, count in nuclearities.most_common():
    print(f'{nuclearity}: {count}')

RST RELATIONS

Joint: 148044
Background: 19982
Elaboration: 16990
Temporal: 5415
Topic-Change: 1722
Same-Unit: 1244
Attribution: 741
TextualOrganization: 669
Evaluation: 589
Contrast: 566
Explanation: 455
Enablement: 426
Cause: 377
Topic-Comment: 102
Condition: 66
Summary: 64
Manner-Means: 52
Comparison: 9

NUCLEARITY

NN: 157554
NS: 32250
SN: 7709
